In [2]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from  tensorflow.python import keras
from tensorflow.python.keras import Sequential
import matplotlib.pyplot as plt
from tensorflow.python.keras.layers import Dense, Activation, Dropout, Flatten, Conv2D, MaxPooling2D
#from tensorflow.python.keras.layers.normalization import BatchNormalization
import numpy as np


shape = (224,224,3)

#train
flat = np.load('flat.npy')
gable = np.load('gable.npy')
hip = np.load('hip.npy')

flat_Y = np.zeros(400,dtype=int)
gable_Y = np.ones(400,dtype=int)
hip_Y = np.zeros(400,dtype=int)
hip_Y[:]=2

X_all = np.vstack((flat[1:],gable[1:],hip[1:]))
X_all = np.expand_dims(X_all, axis=4)

Y_all = np.hstack((flat_Y,gable_Y,hip_Y))

Y_temp = Y_all
res = np.zeros((Y_all.size,3))
res[np.arange(Y_all.size), Y_all] = 1
Y_all=res

X_all=np.squeeze(X_all)

#test
flat_test = np.load('flat_test.npy')
gable_test = np.load('gable_test.npy')
hip_test = np.load('hip_test.npy')

flat_Y_test = np.zeros(2400,dtype=int)
gable_Y_test = np.ones(2400,dtype=int)
hip_Y_test = np.zeros(2400,dtype=int)
hip_Y_test[:]=2

X_all_test = np.vstack((flat_test[1:],gable_test[1:],hip_test[1:]))
X_all_test = np.expand_dims(X_all_test, axis=4)

Y_all_test = np.hstack((flat_Y_test,gable_Y_test,hip_Y_test))

Y_temp_test = Y_all_test
res = np.zeros((Y_all_test.size,3))
res[np.arange(Y_all_test.size), Y_all_test] = 1
Y_all_test=res

X_all_test=np.squeeze(X_all_test)
tx,ty = Y_all.shape

ti = X_all.shape
print(ti)
tx,ty = Y_all.shape
print(tx)

model = Sequential()

# 1st Convolutional Layer
model.add(Conv2D(filters=96, input_shape=(224,224,3), kernel_size=(11,11), strides=(4,4), padding='valid'))
model.add(Activation('relu'))
# Max Pooling
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# 2nd Convolutional Layer
model.add(Conv2D(filters=256, kernel_size=(11,11), strides=(1,1), padding='valid'))
model.add(Activation('relu'))
# Max Pooling
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# 3rd Convolutional Layer
model.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='valid'))
model.add(Activation('relu'))

# 4th Convolutional Layer
model.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='valid'))
model.add(Activation('relu'))

# 5th Convolutional Layer
model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), padding='valid'))
model.add(Activation('relu'))
# Max Pooling
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# Passing it to a Fully Connected layer
model.add(Flatten())
# 1st Fully Connected Layer
model.add(Dense(4096, input_shape=(224*224*3,)))
model.add(Activation('relu'))
# Add Dropout to prevent overfitting
model.add(Dropout(0.4))

# 2nd Fully Connected Layer
model.add(Dense(4096))
model.add(Activation('relu'))
# Add Dropout
model.add(Dropout(0.4))

# 3rd Fully Connected Layer
model.add(Dense(1000))
model.add(Activation('relu'))
# Add Dropout
model.add(Dropout(0.4))

# Output Layer
model.add(Dense(3))
model.add(Activation('softmax'))

model.summary()

# Compile the model
model.compile(loss=keras.losses.categorical_crossentropy, optimizer='adam', metrics=["accuracy"])

# Model configuration
batch_size = 30
no_epochs = 50
no_classes = 3    
verbosity = 1

learning_rate = 0.001

from sklearn import metrics
from sklearn.model_selection import KFold

Acc_all = []
Loss_all = []

Y_Pred_all = np.zeros(tx)
from numpy import argmax
 
# Fit data to model
history = model.fit(X_all, Y_all,
          batch_size=batch_size,
          epochs=no_epochs,
          verbose=verbosity)

score = model.evaluate(X_all_test, Y_all_test,batch_size=1, verbose=1)
print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')

Acc_all.append(score[1])
Loss_all.append(score[0])


Y_Pred = model.predict(X_all_test,batch_size=1,verbose=1)

Y_Pred = argmax(Y_Pred, axis=1)


print(np.average(Acc_all))
print(np.average(Loss_all))

Y_True = Y_temp


import sklearn.metrics as metrics
acc = metrics.accuracy_score(Y_True, Y_Pred_all, normalize=False)
f1 = metrics.f1_score(Y_True,Y_Pred_all, average=None)
precision = metrics.precision_score(Y_True,Y_Pred_all, average=None)
recall = metrics.recall_score(Y_True,Y_Pred_all, average=None)
confusion = metrics.confusion_matrix(Y_True,Y_Pred_all)

print("Accuracy:  "+str(acc))
print("F1:  "+str(f1))
print("Precision:  "+str(precision))
print("Recall:  "+str(recall))
print("Confusion:  "+str(confusion))


FileNotFoundError: ignored

# New Section

In [ ]:
X_all=np.squeeze(X_all)
X_all.shape

(6, 224, 224, 3)

In [ ]:
import os,cv2
import numpy as np

train_path = 'evaluation/GableResized1'
loaded_images = np.zeros((1,224,224,3))


for i in os.listdir(train_path):
  img=cv2.imread(train_path+'/'+ i)
  img=cv2.resize(img,(224,224))
  img=np.expand_dims(img,axis=0)
  loaded_images=np.concatenate((loaded_images,img),axis=0)

np.save('gable_test.npy',loaded_images)